In [1]:
## Step 1: Check Required Files

import os, shutil

OUTPUT_PATH = '/kaggle/working'
MODELS_DIR  = '/kaggle/working/models'
APP_PATH    = os.path.join(OUTPUT_PATH, '09_streamlit_app.py')

os.makedirs(MODELS_DIR, exist_ok=True)

MODEL_PATH = os.path.join(MODELS_DIR,  'best_model.h5')
CLASS_MAP  = os.path.join(OUTPUT_PATH, 'class_mapping.json')

# ── Auto-find files from /kaggle/input/ if missing in /kaggle/working/ ──
def find_in_input(filename):
    for root, dirs, files in os.walk('/kaggle/input'):
        if filename in files:
            return os.path.join(root, filename)
    return None

print("Checking required files...\n")

# best_model.h5
if not os.path.exists(MODEL_PATH):
    src = find_in_input('best_model-2.h5')
    if src:
        shutil.copy(src, MODEL_PATH)
        print(f"   ✓ best_model.h5 copied from: {src}")
    else:
        print("   ✗ best_model.h5 NOT FOUND anywhere!")
        print("     → Add your trained model dataset via Notebook > Add Data")
else:
    size = os.path.getsize(MODEL_PATH) / (1024*1024)
    print(f"   ✓ best_model.h5 ({size:.1f} MB)")

# class_mapping.json
if not os.path.exists(CLASS_MAP):
    src = find_in_input('class_mapping.json')
    if src:
        shutil.copy(src, CLASS_MAP)
        print(f"   ✓ class_mapping.json copied from: {src}")
    else:
        # Auto-generate from dataset if possible
        print("   ⚠️  class_mapping.json not found — will auto-generate...")
        dataset_src = None
        for root, dirs, _ in os.walk('/kaggle/input'):
            if 'train' in dirs:
                dataset_src = root
                break
        if dataset_src:
            train_path = os.path.join(dataset_src, 'train')
            classes = sorted([d for d in os.listdir(train_path)
                              if os.path.isdir(os.path.join(train_path, d))])
            mapping = {
                'class_to_index': {c: i for i, c in enumerate(classes)},
                'index_to_class': {str(i): c for i, c in enumerate(classes)},
                'num_classes': len(classes)
            }
            import json
            with open(CLASS_MAP, 'w') as f:
                json.dump(mapping, f, indent=4)
            print(f"   ✓ class_mapping.json auto-generated — {len(classes)} classes")
        else:
            print("   ✗ class_mapping.json NOT FOUND and cannot be generated!")
else:
    size = os.path.getsize(CLASS_MAP) / 1024
    print(f"   ✓ class_mapping.json ({size:.1f} KB)")

# Final check
print()
all_ok = os.path.exists(MODEL_PATH) and os.path.exists(CLASS_MAP)
if all_ok:
    print("✓ All files ready — proceed to next cells!")
else:
    print("⚠️  Some files still missing. See instructions above.")

Checking required files...

   ✓ best_model.h5 copied from: /kaggle/input/models/kevinchovatiya/skin-disease-efficientnetb3/tensorflow2/default/1/best_model-2.h5
   ✓ class_mapping.json copied from: /kaggle/input/notebooks/kevinchovatiya/06-save-to-drive-ipynb/class_mapping.json

✓ All files ready — proceed to next cells!


In [2]:
## Step 2: Install Required Libraries

import subprocess
subprocess.run(['pip', 'install', 'streamlit', '-q'], check=True)
subprocess.run(['pip', 'install', 'pyngrok',   '-q'], check=True)
subprocess.run(['pip', 'install', 'pandas',    '-q'], check=True)

print("✓ All libraries installed!")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 69.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 116.0 MB/s eta 0:00:00
✓ All libraries installed!


In [3]:
## Step 3: Write Streamlit App File

import shutil, os

# Copy app.py from your input dataset
shutil.copy(
    '/kaggle/input/datasets/kevinchovatiya/dermai-app/app.py',
    '/kaggle/working/09_streamlit_app.py'
)

# Copy model from your input dataset
shutil.copy(
    '/kaggle/input/models/kevinchovatiya/skin-disease-efficientnetb3/tensorflow2/default/1/best_model-2.h5',
    '/kaggle/working/best_model.h5'
)

print("✅ Files copied!")
print(f"   app.py  : {os.path.exists('/kaggle/working/09_streamlit_app.py')}")
print(f"   model   : {os.path.exists('/kaggle/working/best_model.h5')}")

✅ Files copied!
   app.py  : True
   model   : True


In [4]:
## Step 4: Configure ngrok

import subprocess
subprocess.run(['pip', 'install', 'pyngrok', '-q'], check=True)

from pyngrok import ngrok

NGROK_TOKEN = "39vgmgbSCJLkPuxTtefQzZQbi3i_2BxVYujysWoCdr6zjn9eX"
ngrok.set_auth_token(NGROK_TOKEN)
print("✓ ngrok configured!")

✓ ngrok configured!


In [5]:
## Step 5: Launch Streamlit App

import subprocess, threading, time, os
from pyngrok import ngrok

# ── Define path directly here (don't rely on other cells) ──
APP_PATH = '/kaggle/working/09_streamlit_app.py'

# ── Verify app file exists ──
if not os.path.exists(APP_PATH):
    print(f"❌ App file not found: {APP_PATH}")
    print("   Run Cell 3 first to copy the app file!")
else:
    print(f"✅ App file found: {APP_PATH}")

# ── Verify model exists ──
MODEL_PATH = '/kaggle/working/best_model.h5'
if not os.path.exists(MODEL_PATH):
    print(f"❌ Model not found: {MODEL_PATH}")
    print("   Make sure best_model.h5 is attached as input dataset!")
else:
    size = os.path.getsize(MODEL_PATH) / (1024*1024)
    print(f"✅ Model found: {size:.1f} MB")

# ── Kill any old streamlit ──
subprocess.run(['pkill', '-f', 'streamlit'], capture_output=True)
time.sleep(3)

# ── Start streamlit ──
def run_streamlit():
    subprocess.run([
        'streamlit', 'run', APP_PATH,
        '--server.port', '8501',
        '--server.headless', 'true',
        '--server.enableCORS', 'false',
        '--server.enableXsrfProtection', 'false',
        '--server.address', '0.0.0.0',
    ])

threading.Thread(target=run_streamlit, daemon=True).start()
print("\n⏳ Waiting for Streamlit to start...")
time.sleep(8)   # ← wait longer

# ── Check if streamlit is running ──
result = subprocess.run(['pgrep', '-f', 'streamlit'], capture_output=True, text=True)
if result.returncode == 0:
    print("✅ Streamlit is running!")
else:
    print("❌ Streamlit failed to start — check app file for errors")

# ── Connect ngrok ──
public_url = ngrok.connect(8501)

print("\n" + "="*60)
print("🎉 APP IS LIVE!")
print("="*60)
print(f"\n🌐 URL: {public_url}")
print("\n👆 Click the URL to open your app!")
print("⚠️  Keep this cell running")
print("="*60)

✅ App file found: /kaggle/working/09_streamlit_app.py
✅ Model found: 131.6 MB

⏳ Waiting for Streamlit to start...



  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.19.2.2:8501
  External URL: http://136.112.231.126:8501

✅ Streamlit is running!

🎉 APP IS LIVE!

🌐 URL: NgrokTunnel: "https://agronomical-noncandescently-katharine.ngrok-free.dev" -> "http://localhost:8501"

👆 Click the URL to open your app!
⚠️  Keep this cell running
